# Clusters That Are Not Round

**Topic:** Gaussian mixtures &nbsp;·&nbsp; **Difficulty:** Medium

k-means minimizes squared Euclidean distance, which is the same as assuming every
cluster is a sphere of equal size. Hand it two long thin diagonal clusters and it
will slice them the wrong way, because the nearest *centroid* is not the nearest
*cluster* once the shapes are elongated.

A Gaussian mixture with full covariance has no such assumption. Fit both and
compare against the labels that generated the data.

## The data (created in the next cell)

- `X` `(1000, 2)`: two elongated, diagonally oriented Gaussian clusters.
- `y_true` `(1000,)`: the generating component, used only for scoring.

## Your task

With **sklearn and NumPy**:

1. Fit `KMeans(n_clusters=2, n_init=10, random_state=0)`.
2. Fit `GaussianMixture(n_components=2, covariance_type="full", random_state=0,
   n_init=10, init_params="random_from_data")`. The `init_params` matters: the
   default initializes from k-means, which hands the mixture the very local
   optimum you are trying to escape.
3. Cluster labels are arbitrary, so score with **adjusted Rand index**
   (`sklearn.metrics.adjusted_rand_score`), which is invariant to relabelling.
4. Also fit a spherical mixture with the **same** initialization settings, to
   confirm it is the covariance shape, not the algorithm or the initialization,
   that matters.
5. Set `answers` to a dict with `"ari_kmeans"`, `"ari_gmm_full"`,
   `"ari_gmm_spherical"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the full-covariance mixture beats both of the spherical methods by a wide
margin.

In [ ]:
import numpy as np

rng = np.random.default_rng(1717)
n = 500
theta = 0.7
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
stretch = np.diag([4.0, 0.35])
short_axis = R @ np.array([0.0, 1.0])      # perpendicular to the elongation
offset = 1.6 * short_axis
a = rng.normal(0, 1, (n, 2)) @ stretch @ R.T - offset
b = rng.normal(0, 1, (n, 2)) @ stretch @ R.T + offset
X = np.vstack([a, b])
y_true = np.concatenate([np.zeros(n, int), np.ones(n, int)])
print(f"X {X.shape}: two parallel cigars, length std 4.0, width std 0.35, "
      f"separated by {2 * 1.6:.1f} across their width")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("ari_kmeans", [], -0.00099308530449099, 0.00016628023192542379, 1e-06),
    ("ari_gmm_full", [], 1.0, -0.16743801481449916, 1e-06),
    ("ari_gmm_spherical", [], -0.0009503355072698923, 0.0001591222907450008, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "ari_gmm_full" in answers:
    _w = (answers["ari_gmm_full"] > answers["ari_kmeans"] + 0.3
          and answers["ari_gmm_full"] > answers["ari_gmm_spherical"] + 0.3)
    print(f"  {'full covariance wins':<24} {'ok' if _w else 'wrong'}  "
          f"(full {answers['ari_gmm_full']:.3f}, kmeans {answers['ari_kmeans']:.3f}, "
          f"spherical {answers['ari_gmm_spherical']:.3f})")
    _ok = _ok and _w

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

k-means' decision boundary is the perpendicular bisector between its own two centroids,
always a straight line at right angles to the line joining them. The true
boundary here runs *along* the elongation, but k-means' objective prefers to split
each long cigar across its length, which pulls its centroids out along the
elongation. Their bisector then cuts both cigars in half
instead of separating them. A full-covariance mixture learns the orientation and
scale of each component, so its boundary is a conic that follows the geometry.
The spherical mixture reproduces k-means' failure with identical initialization
settings, which isolates the cause as the covariance shape rather than the
algorithm. Watch the `init_params` too: seeding EM from k-means hands it the bad
local optimum and it never leaves.

```python
import numpy as np
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
gm = GaussianMixture(n_components=2, covariance_type="full", random_state=0,
                     n_init=10, init_params="random_from_data").fit(X)
gs = GaussianMixture(n_components=2, covariance_type="spherical", random_state=0,
                     n_init=10, init_params="random_from_data").fit(X)

answers = {
    "ari_kmeans": float(adjusted_rand_score(y_true, km.labels_)),
    "ari_gmm_full": float(adjusted_rand_score(y_true, gm.predict(X))),
    "ari_gmm_spherical": float(adjusted_rand_score(y_true, gs.predict(X))),
}
```